In [2]:
import utils
import numpy as np

In [ ]:
# ==================
# GENERAL PARAMETERS
# ==================

n = 5
steps = 10               # number of steps required for time evolution

def u(x):
    return np.sin(2*np.pi*2*x) + 0.5*np.sin(2*np.pi*7*x)

nu = 1e-3                 # diffusion coefficient 
save_every = 200          # after this many steps, take a snapshot of the function to plot on the graph
cfl = 0.1                 # controls time step relative to grid spacing. affects stability of time0-step scheme



# =============
# TN PARAMETERS
# =============

# parameters for forming the initial MPS
init_cutoff = 1e-12
init_max_bond = 64

# parameters for forming the initial MPO
mpo_cutoff = 1e-12
mpo_max_bond = 256

In [4]:
N     = 2**n
x     = np.linspace(0, 1, N, endpoint=False)

dx    = x[1] - x[0] 
dt    = cfl * dx*dx / nu 
u0    = u(x)
alpha = nu * dt / (dx * dx)

In [5]:

L = utils.laplacian_1d(N, dx, "dirichlet", "dense")

A_dense = np.eye(N) + dt * nu * L                  # your dense time-step matrix
A_svd = utils.mat_to_qtt_mpo(A_dense, n, mpo_cutoff, mpo_max_bond)
A_manual = utils.qtt_diffusion_mpo(n, alpha, mpo_cutoff, mpo_max_bond)

A_svd_dense = np.asarray(A_svd.to_dense()).reshape(N, N)
A_manual_dense = np.asarray(A_manual.to_dense()).reshape(N, N)

print("||A_svd - A_dense|| =", np.linalg.norm(A_svd_dense - A_dense))
print("||A_manual - A_dense|| =", np.linalg.norm(A_manual_dense - A_dense))
print("||A_manual - A_svd||  =", np.linalg.norm(A_manual_dense - A_svd_dense))

||A_svd - A_dense|| = 4.27863164248793e-15
||A_manual - A_dense|| = 1.1135528725660044
||A_manual - A_svd||  = 1.113552872566004
